# ДЗ 2. Функции и словари на настоящих данных
Срок: **до понедельника 19:00**.

**Сдача через GitHub** (инструкция «Git для домашних заданий»): сохраните ноутбук как `hw02.ipynb` в папку `hw02` репозитория `dsa-homework`, затем `git add hw02` → `git commit -m "hw02: ..."` → `git push`. Из Colab: Файл → Скачать → Скачать .ipynb. Файл `orders.csv` в репозиторий класть не нужно.

Файл `orders.csv` — заказы «Дала Маркет» за сентябрь и октябрь 2026. Работаем **без Pandas и NumPy**: только `csv`, функции, словари, множества.

- **[База]** обязательно для всех.
- **[Стандарт]** решайте, сколько получится.
- **[Челлендж]** для тех, кому легко.

**Правило AI:** можно спросить, *почему* не работает код, но нельзя просить решение.

## [Для всех] Чтение кода — без запуска
Для каждого фрагмента: **сначала** напишите в Markdown-ячейке ответ и одно предложение «почему», **потом** запустите и сравните. Ошибка в предсказании не снижает оценку — снижает отсутствие объяснения.
```python
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)
```
На следующем занятии несколько человек объяснят вслух одну строку своего ДЗ.

Мои предсказания и объяснения:

1.
2.
3.
4.
5.

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [2]:
import csv
with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "строк")

5463 строк


## [База]
1. Напишите функции `to_num(s)` и `clean_city(s)` (6 городов, латиница и разный регистр → кириллица с заглавной).
2. Сколько заказов в каждой категории (`category`)? Отсортируйте по убыванию.
3. Для каждой категории посчитайте выручку оплаченных заказов и средний чек. Выведите таблицу f-строкой.
4. Сколько заказов в каждом месяце? Месяц — это `order_date[:7]`.

In [3]:
# База
# 1) Функции to_num(s) и clean_city(s)
import csv
from collections import Counter, defaultdict
def to_num(s):
    """Преобразует строку в число; пустое или некорректное значение -> None."""
    if s is None:
        return None
    s = str(s).strip().replace("\u00a0", "").replace(" ", "").replace(",", ".")
    if not s:
        return None
    try:
        return float(s)
    except ValueError:
        return None
CITY_MAP = {
    "алматы": "Алматы", "almaty": "Алматы",
    "astana": "Астана", "астана": "Астана",
    "shymkent": "Шымкент", "шымкент": "Шымкент",
    "karaganda": "Караганда", "караганда": "Караганда",
    "aktobe": "Актобе", "актобе": "Актобе",
    "atyrau": "Атырау", "атырау": "Атырау",
}

def clean_city(s):
    """Нормализует 6 городов: латиница/кириллица, регистр и пробелы."""
    if s is None:
        return ""
    key = str(s).strip().casefold()
    return CITY_MAP.get(key, str(s).strip().title())
with open("orders.csv", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f))

def is_paid(row):
    return row["status"].strip().casefold() == "оплачен"

def is_cancelled(row):
    return row["status"].strip().casefold() == "отменён"

print(f"Загружено строк: {len(rows)}")

Загружено строк: 5463


In [4]:
# 2. Количество заказов в каждой категории (по убыванию)
category_counts = Counter(row["category"] for row in rows)
print("Заказы по категориям:")
for category, count in category_counts.most_common():
    print(f"{category:15} {count:5}")

Заказы по категориям:
Одежда           1529
Электроника      1185
Дом              1105
Красота           991
Книги             653


In [19]:
# 3. Выручка оплаченных заказов и средний чек по категориям
category_revenue = defaultdict(float)
category_paid_count = Counter()
for row in rows:
    if is_paid(row):
        amount = to_num(row["amount"])
        if amount is not None:
            category_revenue[row["category"]] += amount
            category_paid_count[row["category"]] += 1
print("Категория        Выручка          Средний чек")
for category in sorted(category_revenue):
    revenue = category_revenue[category]
    avg = revenue / category_paid_count[category]
    print(f"{category:15} {revenue:14,.2f} {avg:14,.2f}")

Категория        Выручка          Средний чек
Дом              10,516,320.00      10,654.83
Книги             2,294,100.00       3,934.99
Красота           5,507,620.00       6,188.34
Одежда           18,321,080.00      13,285.77
Электроника      65,270,240.00      61,286.61


In [18]:
# 4. Количество заказов по месяцам
month_counts = Counter(row["order_date"][:7] for row in rows)
print("Заказы по месяцам:")
for month in sorted(month_counts):
    print(f"{month}: {month_counts[month]}")

Заказы по месяцам:
2026-09: 2782
2026-10: 2681


## [Стандарт]
5. Выручка оплаченных по месяцам и изменение октября к сентябрю в процентах.
6. Доля отменённых заказов по каждому (чистому) городу, в процентах с одним знаком.
7. Топ-5 клиентов (`customer_id`) по выручке оплаченных заказов. Кто на первом месте и что с ним не так? Ответьте в Markdown-ячейке.
8. Сколько **уникальных** клиентов в каждом городе? Подсказка: словарь множеств.

In [17]:
# Стандарт
# 5) Выручка оплаченных заказов по месяцам и изменение октября к сентябрю
month_revenue = defaultdict(float)
for row in rows:
    if is_paid(row):
        amount = to_num(row["amount"])
        if amount is not None:
            month_revenue[row["order_date"][:7]] += amount

print("Выручка оплаченных заказов:")
for month in sorted(month_revenue):
    print(f"{month}: {month_revenue[month]:,.2f}")

sep_revenue = month_revenue.get("2026-09", 0)
oct_revenue = month_revenue.get("2026-10", 0)
change_pct = (oct_revenue - sep_revenue) / sep_revenue * 100 if sep_revenue else 0
print(f"Изменение октября к сентябрю: {change_pct:+.2f}%")

Выручка оплаченных заказов:
2026-09: 57,489,850.00
2026-10: 44,419,510.00
Изменение октября к сентябрю: -22.74%


In [8]:
# 6. Доля отменённых заказов по нормализованным городам
city_status = defaultdict(lambda: {"cancelled": 0, "total": 0})
for row in rows:
    city = clean_city(row["city"])
    city_status[city]["total"] += 1
    if is_cancelled(row):
        city_status[city]["cancelled"] += 1

print("Доля отмен по городам:")
for city in sorted(city_status):
    data = city_status[city]
    share = data["cancelled"] / data["total"] * 100 if data["total"] else 0
    print(f"{city:12} {share:5.1f}% ({data['cancelled']}/{data['total']})")

Доля отмен по городам:
Актобе        10.3% (41/397)
Алматы         7.9% (172/2188)
Астана         8.8% (123/1390)
Атырау         7.5% (11/146)
Караганда      8.9% (48/538)
Шымкент       20.4% (164/804)


In [20]:
# 7. Топ-5 клиентов по выручке оплаченных заказов
customer_revenue = defaultdict(float)
for row in rows:
    if is_paid(row):
        amount = to_num(row["amount"])
        if amount is not None:
            customer_revenue[row["customer_id"]] += amount

top5_customers = sorted(customer_revenue.items(), key=lambda item: item[1], reverse=True)[:5]
print("Топ-5 клиентов:")
for customer, revenue in top5_customers:
    print(f"{customer:10} {revenue:,.2f}")

Топ-5 клиентов:
TEST       29,999,970.00
C01627     257,180.00
C00485     215,920.00
C01338     192,080.00
C02122     191,670.00


In [10]:
# На первом месте находится клиент с аномально высокой суммой выручки. Может быть это связано с технической ошибкой

In [21]:
# 8. Уникальные клиенты в каждом городе (словарь множеств)
city_customers = defaultdict(set)
for row in rows:
    city_customers[clean_city(row["city"])].add(row["customer_id"])

print("Уникальные клиенты по городам:")
for city in sorted(city_customers):
    print(f"{city:12} {len(city_customers[city])}")

Уникальные клиенты по городам:
Актобе       358
Алматы       1351
Астана       981
Атырау       138
Караганда    451
Шымкент      655


## [Челлендж]
9. Найдите полные дубли строк: превратите каждую строку в `tuple(r.values())` и проверяйте через множество. Сколько их и в каком месяце?
10. Пересчитайте выручку по месяцам без дублей и без клиента из задачи 7. Как изменился процент из задачи 5?
11. У какого клиента больше всего заказов (после удаления дублей)? Сколько?
12. Напишите менеджеру 3–4 предложения: насколько на самом деле упала выручка и почему первый расчёт был неверным.

In [12]:
# Челлендж
# 9. Полные дубли строк
seen_rows = set()
duplicate_rows = []
unique_rows = []
for row in rows:
    row_tuple = tuple(row.values())
    if row_tuple in seen_rows:
        duplicate_rows.append(row)
    else:
        seen_rows.add(row_tuple)
        unique_rows.append(row)

duplicate_months = Counter(row["order_date"][:7] for row in duplicate_rows)
print(f"Найдено полных дублей: {len(duplicate_rows)}")
for month in sorted(duplicate_months):
    print(f"{month}: {duplicate_months[month]}")

Найдено полных дублей: 180
2026-09: 180


In [13]:
# 10. Выручка по месяцам после удаления дублей и клиента-лидера из задания 7
top_customer = top5_customers[0][0]
clean_month_revenue = defaultdict(float)
for row in unique_rows:
    if is_paid(row) and row["customer_id"] != top_customer:
        amount = to_num(row["amount"])
        if amount is not None:
            clean_month_revenue[row["order_date"][:7]] += amount

print(f"Исключён клиент: {top_customer}")
for month in sorted(clean_month_revenue):
    print(f"{month}: {clean_month_revenue[month]:,.2f}")

clean_sep = clean_month_revenue.get("2026-09", 0)
clean_oct = clean_month_revenue.get("2026-10", 0)
clean_change_pct = (clean_oct - clean_sep) / clean_sep * 100 if clean_sep else 0
print(f"Изменение октября к сентябрю после очистки: {clean_change_pct:+.2f}%")
print(f"Первоначальное изменение: {change_pct:+.2f}%")
print(f"Изменение показателя: {clean_change_pct-change_pct:+.2f} п.п.")

Исключён клиент: TEST
2026-09: 35,243,410.00
2026-10: 34,419,520.00
Изменение октября к сентябрю после очистки: -2.34%
Первоначальное изменение: -22.74%
Изменение показателя: +20.40 п.п.


In [14]:
# 11. Клиент с наибольшим количеством заказов после удаления дублей
customer_order_counts = Counter(row["customer_id"] for row in unique_rows)
most_orders_customer, most_orders_count = customer_order_counts.most_common(1)[0]
print(f"Больше всего заказов у {most_orders_customer}: {most_orders_count}")

Больше всего заказов у C01818: 10


In [22]:
print("Сообщение менеджеру:")
print(
    f"Первоначальный расчёт показал изменение выручки октября к сентябрю "
    f"на {change_pct:.2f}%. После удаления {len(duplicate_rows)} полных дублей "
    f"и исключения аномального клиента {top_customer} изменение составило "
    f"{clean_change_pct:.2f}%. Клиент {top_customer} дал непропорционально большую "
    f"выручку ({top5_customers[0][1]:,.2f}), поэтому исказил общую картину. "
    f"Первый расчёт был завышен повторяющимися строками и аномальной записью клиента; "
    f"для управленческих выводов следует использовать очищенные данные."
)

Сообщение менеджеру:
Первоначальный расчёт показал изменение выручки октября к сентябрю на -22.74%. После удаления 180 полных дублей и исключения аномального клиента TEST изменение составило -2.34%. Клиент TEST дал непропорционально большую выручку (29,999,970.00), поэтому исказил общую картину. Первый расчёт был завышен повторяющимися строками и аномальной записью клиента; для управленческих выводов следует использовать очищенные данные.
